# ResearchLanka Kaggle Pipeline With Dagster, No Data Collection

This notebook runs the Dagster preprocessing pipeline from already-uploaded raw/source files. It does **not** call the data collection APIs or harvesters.

It expects your Kaggle dataset to contain:

```text
backend/data/raw/
backend/data/processed/crossref/
```

Default Kaggle input path used below:

```text
/kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data/backend/data
```

If your Kaggle dataset path changes, edit `DATASET_DATA_DIR` in the setup cell.

## 1. Setup Paths

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/krish-anu/researchlanka-ai.git"
WORK_DIR = Path("/kaggle/working")
CODE_DIR = WORK_DIR / "code"
BACKEND_DIR = CODE_DIR / "backend"
DAGSTER_DIR = BACKEND_DIR / "dagster-quickstart"
DATASET_DATA_DIR = Path("/kaggle/input/datasets/anusankrishnathas/researchlanka-raw-data/backend/data")

print("Code dir:", CODE_DIR)
print("Backend dir:", BACKEND_DIR)
print("Dagster dir:", DAGSTER_DIR)
print("Dataset data dir:", DATASET_DATA_DIR)

## 2. Clone Latest Code

In [ ]:
%cd /kaggle/working
!rm -rf code
!git clone {REPO_URL} code
%cd /kaggle/working/code/backend
!ls scripts src dagster-quickstart requirements.txt

## 3. Copy Uploaded Raw Data Into The Cloned Backend

In [ ]:
%cd /kaggle/working/code/backend
!mkdir -p data
!cp -r {DATASET_DATA_DIR}/* data/
!find data -maxdepth 3 -type f | head -50

## 4. Install Dependencies

In [ ]:
%cd /kaggle/working/code/backend
!pip install $(grep -v '^psycopg2==' requirements.txt)
!pip install -e dagster-quickstart

## 5. Run Dagster Preprocessing Job Without Collection

This job prepares source CSVs from existing files, then runs the common merge, deduplication, final dataset, year filter, language normalization, multivalue normalization, and analysis-ready dataset assets.

It does not run OpenAlex/Crossref/SLJOL API collection or repository harvesting.

In [ ]:
%cd /kaggle/working/code/backend/dagster-quickstart
!dagster job execute \
  -m dagster_quickstart.definitions \
  -j researchlanka_no_collection_preprocessing_job

## 6. Verify Dagster Outputs

In [ ]:
%cd /kaggle/working/code/backend
!ls -lh data/processed/repositories_combined.csv data/processed/sljol.csv
!ls -lh data/processed/common/common_publications_final.csv
!ls -lh data/processed/common/common_publications_final_2016_2026_analysis_ready.csv

## 7. Build Embeddings And Train Models

These modeling steps are currently Make/Python script steps, not Dagster assets.

In [ ]:
%cd /kaggle/working/code/backend
!make model-embeddings PYTHON=python
!make train-logreg PYTHON=python

## 8. Train Linear SVM

In [ ]:
%cd /kaggle/working/code/backend
!python scripts/modeling/train_linear_svm_classifier.py \
  --input data/processed/common/common_publications_final.csv \
  --label-column primary_domain \
  --text-columns title,abstract,keywords \
  --ngram-max 2 \
  --max-features 30000 \
  --c-values 0.1,1,10 \
  --cv-folds 3

## 9. Compare Model Metrics

In [ ]:
%cd /kaggle/working/code/backend
!grep -E "model_family|label_column|accuracy|macro_f1|weighted_f1" data/models/*metrics.txt || true

## 10. Zip Outputs For Download

In [ ]:
%cd /kaggle/working/code/backend
!zip -r /kaggle/working/researchlanka-kaggle-outputs.zip data/processed data/models
!ls -lh /kaggle/working/researchlanka-kaggle-outputs.zip

Download this file from Kaggle output:

```text
/kaggle/working/researchlanka-kaggle-outputs.zip
```